<a href="https://colab.research.google.com/github/palamsaiteja333/agent-systems-engineering/blob/main/02_smolagents_first_tool_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "smolagents[toolkit]" huggingface_hub pytz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HuggingFaceAgentBuilding")
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token loaded successfully")

HF token loaded successfully


In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, load_tool, tool
import datetime
import requests
import pytz
import yaml

In [9]:
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

print("Model configured")

Model configured


In [10]:
@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """
    Get the current local time in a specified timezone.

    Args:
        timezone: A valid timezone such as America/Toronto,
                  America/Edmonton, Asia/Kolkata, or Asia/Tokyo.
    """

    try:
        tz = pytz.timezone(timezone)

        local_time = datetime.datetime.now(tz).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        return f"The current local time in {timezone} is {local_time}"

    except Exception as e:
        return f"Unable to get time for '{timezone}': {str(e)}"

In [11]:
get_current_time_in_timezone("Asia/Tokyo")

'The current local time in Asia/Tokyo is 2026-09-21 07:06:57'

In [12]:
agent = CodeAgent(
    model=model,
    tools=[get_current_time_in_timezone],
    max_steps=6
)

print("Agent created")

Agent created


In [13]:
response = agent.run(
    "What time is it currently in Tokyo?"
)

print(response)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What time is it currently in Tokyo?                                                                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tokyo_time = get_current_time_in_timezone(timezone="Asia/Tokyo")                                                 
  print("Current time in Tokyo:", tokyo_time)                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current time in Tokyo: The current local time in Asia/Tokyo is 2026-09-21 07:07:55

Out: None

[Step 1: Duration 2.30 seconds| Input tokens: 2,091 | Output tokens: 63]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Extracting the time part from the output string                                                                
  tokyo_time_str = "The current local time in Asia/Tokyo is 2026-09-21 07:07:55"                                   
  tokyo_time = tokyo_time_str.split(' ')[-1]                                                                       
  final_answer(tokyo_time)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 07:07:55

[Step 2: Duration 2.61 seconds| Input tokens: 4,370 | Output tokens: 163]

07:07:55


In [15]:
@tool
def lookup_inventory(product_id: str) -> str:
    """
    Retrieve inventory information for a product.

    Args:
        product_id: Product identifier such as SKU-1001.
    """

    inventory = {
        "SKU-1001": {
            "product": "Industrial Sensor",
            "quantity": 14,
            "warehouse": "Calgary"
        },
        "SKU-1002": {
            "product": "Control Module",
            "quantity": 0,
            "warehouse": "Edmonton"
        },
        "SKU-1003": {
            "product": "Power Supply",
            "quantity": 42,
            "warehouse": "Toronto"
        }
    }

    product = inventory.get(product_id)

    if product is None:
        return f"No inventory record found for {product_id}"

    return str(product)

In [16]:
lookup_inventory("SKU-1002")

"{'product': 'Control Module', 'quantity': 0, 'warehouse': 'Edmonton'}"

In [17]:
@tool
def get_order_status(order_id: str) -> str:
    """
    Retrieve the current status of an enterprise customer order.

    Args:
        order_id: Order identifier such as ORD-100.
    """

    orders = {
        "ORD-100": {
            "status": "Processing",
            "product_id": "SKU-1001"
        },
        "ORD-101": {
            "status": "Shipped",
            "product_id": "SKU-1003"
        },
        "ORD-102": {
            "status": "Delayed",
            "product_id": "SKU-1002"
        }
    }

    order = orders.get(order_id)

    if order is None:
        return f"No order found for {order_id}"

    return str(order)

In [18]:
operations_agent = CodeAgent(
    model=model,
    tools=[
        lookup_inventory,
        get_order_status
    ],
    max_steps=6
)

In [19]:
response = operations_agent.run(
        """
    Customer order ORD-102 is delayed.

    Investigate the order and determine whether inventory
    availability could be contributing to the delay.
    """

)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Customer order ORD-102 is delayed.                                                                              │
│                                                                                                                 │
│     Investigate the order and determine whether inventory                                                       │
│     availability could be contributing to the delay.                                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  order_status = get_order_status(order_id="ORD-102")                                                              
  print("Order Status:", order_status)                                                                             
                                                                                                                   
  # Extracting potential product IDs from the order status                                                         
  import re                                                                                                        
  product_ids = re.findall(r'SKU-\d+', order_status)                                                               
  print("Product IDs found in the order:", product_ids)                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Order Status: {'status': 'Delayed', 'product_id': 'SKU-1002'}
Product IDs found in the order: ['SKU-1002']

Out: None

[Step 1: Duration 2.80 seconds| Input tokens: 2,138 | Output tokens: 108]

...

[Step 2: Duration 2.52 seconds| Input tokens: 4,548 | Output tokens: 211]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Inventory")                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Inventory

[Step 3: Duration 2.17 seconds| Input tokens: 7,181 | Output tokens: 299]

In [20]:
operations_agent.run(
    "What is the inventory for SKU-1003?"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the inventory for SKU-1003?                                                                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sku_id = "SKU-1003"                                                                                              
  inventory_info = lookup_inventory(product_id=sku_id)                                                             
  print(inventory_info)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'product': 'Power Supply', 'quantity': 42, 'warehouse': 'Toronto'}

Out: None

[Step 1: Duration 1.97 seconds| Input tokens: 2,119 | Output tokens: 65]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sku_id = "SKU-1003"                                                                                              
  inventory_info = lookup_inventory(product_id=sku_id)                                                             
  final_answer(inventory_info)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'product': 'Power Supply', 'quantity': 42, 'warehouse': 'Toronto'}

[Step 3: Duration 2.44 seconds| Input tokens: 6,952 | Output tokens: 280]

"{'product': 'Power Supply', 'quantity': 42, 'warehouse': 'Toronto'}"

In [21]:
operations_agent.run(
    "Tell me a joke."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me a joke.                                                                                                 │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import random                                                                                                    
                                                                                                                   
  jokes = [                                                                                                        
      "Why was the math book sad? Because it had too many problems.",                                              
      "Why couldn't the bicycle stand up by itself? Because it was two-tired!",                                    
      "Why was the computer cold? It left its Windows open.",                                                      
      "Why don't scientists trust atoms? Because they make up everything!",                                        
      "Why did the tomato turn red? Because it saw the salad dressing!"                                            
  ]                                                                                                                
                                                                                                                   
  joke = random.choice(jokes)                                                                                      
  final_answer(joke)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Why don't scientists trust atoms? Because they make up everything!

[Step 1: Duration 2.96 seconds| Input tokens: 2,112 | Output tokens: 120]

"Why don't scientists trust atoms? Because they make up everything!"